In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA


# ============================================================
# Paste your EMBEDDING_SPACE_JSON log lines here
# ============================================================

LOG_TEXT = r"""
PASTE YOUR LOGS HERE

Example:

EMBEDDING_SPACE_JSON {"type":"embedding_space","round":1,"client":0,...}
EMBEDDING_SPACE_JSON {"type":"embedding_space","round":1000,"client":0,...}
EMBEDDING_SPACE_JSON {"type":"embedding_space","round":2000,"client":1,...}
"""

In [ ]:
# ============================================================
# Parse logs
# ============================================================

records = []

for line in LOG_TEXT.splitlines():

    line = line.strip()

    if not line.startswith("EMBEDDING_SPACE_JSON"):
        continue

    json_string = line[
        len("EMBEDDING_SPACE_JSON "):
    ]

    try:
        record = json.loads(json_string)
        records.append(record)

    except json.JSONDecodeError as e:
        print("Could not parse log line:")
        print(line[:200])
        print(e)


# ============================================================
# Group snapshots by client
# ============================================================

records_by_client = {}

for record in records:

    client = record["client"]

    if client not in records_by_client:
        records_by_client[client] = []

    records_by_client[client].append(record)


# Sort each client's snapshots by round

for client in records_by_client:

    records_by_client[client].sort(
        key=lambda r: r["round"]
    )


# ============================================================
# Display what was found
# ============================================================

print(
    f"Found {len(records)} total snapshots "
    f"from {len(records_by_client)} clients.\n"
)

for client in sorted(records_by_client):

    rounds = [
        record["round"]
        for record in records_by_client[client]
    ]

    print(
        f"Client {client}: "
        f"{len(rounds)} snapshots"
    )

    print(
        f"  Rounds: {rounds}"
    )

In [ ]:
# ============================================================
# Fit one PCA independently for each client
# ============================================================

pca_by_client = {}

for client, client_records in records_by_client.items():

    all_vectors = []

    # --------------------------------------------------------
    # Collect ALL examples and proxies from this client
    # across all rounds where the client was evaluated.
    # --------------------------------------------------------

    for record in client_records:

        for example in record["examples"]:

            all_vectors.append(
                example["embedding"]
            )

        for proxy in record["proxies"]:

            all_vectors.append(
                proxy["embedding"]
            )

    all_vectors = np.asarray(
        all_vectors,
        dtype=np.float32,
    )

    # --------------------------------------------------------
    # Fit PCA
    # --------------------------------------------------------

    pca = PCA(
        n_components=2
    )

    pca.fit(
        all_vectors
    )

    pca_by_client[client] = pca

    print(
        f"Client {client}:"
    )

    print(
        f"  Vectors: {all_vectors.shape[0]}"
    )

    print(
        f"  Dimension: {all_vectors.shape[1]}"
    )

    print(
        f"  PC1 variance: "
        f"{pca.explained_variance_ratio_[0] * 100:.2f}%"
    )

    print(
        f"  PC2 variance: "
        f"{pca.explained_variance_ratio_[1] * 100:.2f}%"
    )

    print(
        f"  Total: "
        f"{pca.explained_variance_ratio_.sum() * 100:.2f}%"
    )

    print()

In [ ]:
# ============================================================
# Select client
# ============================================================

CLIENT = 0


client_records = records_by_client[CLIENT]

pca = pca_by_client[CLIENT]


# ============================================================
# Plot every available round for this client
# ============================================================

for record in client_records:

    round_number = record["round"]

    examples = record["examples"]
    proxies = record["proxies"]

    # --------------------------------------------------------
    # Original high-dimensional embeddings
    # --------------------------------------------------------

    X_examples = np.asarray(
        [
            example["embedding"]
            for example in examples
        ],
        dtype=np.float32,
    )

    y_examples = np.asarray(
        [
            example["label"]
            for example in examples
        ],
        dtype=np.int64,
    )

    X_proxies = np.asarray(
        [
            proxy["embedding"]
            for proxy in proxies
        ],
        dtype=np.float32,
    )

    proxy_labels = np.asarray(
        [
            proxy["label"]
            for proxy in proxies
        ],
        dtype=np.int64,
    )

    # --------------------------------------------------------
    # Project into THIS CLIENT'S PCA space
    # --------------------------------------------------------

    examples_2d = pca.transform(
        X_examples
    )

    proxies_2d = pca.transform(
        X_proxies
    )

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    plt.figure(
        figsize=(10, 8)
    )

    # --------------------------------------------------------
    # Examples
    # --------------------------------------------------------

    classes = sorted(
        np.unique(y_examples)
    )

    for class_id in classes:

        mask = (
            y_examples == class_id
        )

        plt.scatter(
            examples_2d[mask, 0],
            examples_2d[mask, 1],
            s=25,
            alpha=0.45,
            label=f"Class {class_id}",
        )

    # --------------------------------------------------------
    # Personalized proxies
    # --------------------------------------------------------

    plt.scatter(
        proxies_2d[:, 0],
        proxies_2d[:, 1],
        marker="*",
        s=350,
        edgecolors="black",
        linewidths=1.5,
        label="Personalized proxy",
        zorder=10,
    )

    # --------------------------------------------------------
    # Proxy labels
    # --------------------------------------------------------

    for point, label in zip(
        proxies_2d,
        proxy_labels,
    ):

        plt.annotate(
            f"P{label}",
            (
                point[0],
                point[1],
            ),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=11,
            fontweight="bold",
        )

    # --------------------------------------------------------
    # Axis labels
    # --------------------------------------------------------

    pc1 = (
        pca.explained_variance_ratio_[0]
        * 100
    )

    pc2 = (
        pca.explained_variance_ratio_[1]
        * 100
    )

    plt.xlabel(
        f"PC1 ({pc1:.2f}% variance)"
    )

    plt.ylabel(
        f"PC2 ({pc2:.2f}% variance)"
    )

    plt.title(
        f"Embedding Space — "
        f"Client {CLIENT} — "
        f"Round {round_number}"
    )

    plt.grid(
        alpha=0.2
    )

    plt.legend()

    plt.tight_layout()

    plt.show()

In [ ]:
# ============================================================
# Plot every client and every available round
# ============================================================

for client in sorted(records_by_client):

    print(
        f"\n{'=' * 70}"
    )

    print(
        f"CLIENT {client}"
    )

    print(
        f"{'=' * 70}"
    )

    client_records = records_by_client[client]
    pca = pca_by_client[client]

    for record in client_records:

        round_number = record["round"]

        examples = record["examples"]
        proxies = record["proxies"]

        # ----------------------------------------------------
        # Recover original vectors
        # ----------------------------------------------------

        X_examples = np.asarray(
            [
                example["embedding"]
                for example in examples
            ],
            dtype=np.float32,
        )

        y_examples = np.asarray(
            [
                example["label"]
                for example in examples
            ],
            dtype=np.int64,
        )

        X_proxies = np.asarray(
            [
                proxy["embedding"]
                for proxy in proxies
            ],
            dtype=np.float32,
        )

        proxy_labels = np.asarray(
            [
                proxy["label"]
                for proxy in proxies
            ],
            dtype=np.int64,
        )

        # ----------------------------------------------------
        # PCA
        # ----------------------------------------------------

        examples_2d = pca.transform(
            X_examples
        )

        proxies_2d = pca.transform(
            X_proxies
        )

        # ----------------------------------------------------
        # Plot
        # ----------------------------------------------------

        plt.figure(
            figsize=(10, 8)
        )

        classes = sorted(
            np.unique(y_examples)
        )

        for class_id in classes:

            mask = (
                y_examples == class_id
            )

            plt.scatter(
                examples_2d[mask, 0],
                examples_2d[mask, 1],
                s=25,
                alpha=0.45,
                label=f"Class {class_id}",
            )

        # Personalized proxies

        plt.scatter(
            proxies_2d[:, 0],
            proxies_2d[:, 1],
            marker="*",
            s=350,
            edgecolors="black",
            linewidths=1.5,
            label="Personalized proxy",
            zorder=10,
        )

        # Proxy labels

        for point, label in zip(
            proxies_2d,
            proxy_labels,
        ):

            plt.annotate(
                f"P{label}",
                (
                    point[0],
                    point[1],
                ),
                xytext=(8, 8),
                textcoords="offset points",
                fontsize=11,
                fontweight="bold",
            )

        pc1 = (
            pca.explained_variance_ratio_[0]
            * 100
        )

        pc2 = (
            pca.explained_variance_ratio_[1]
            * 100
        )

        plt.xlabel(
            f"PC1 ({pc1:.2f}% variance)"
        )

        plt.ylabel(
            f"PC2 ({pc2:.2f}% variance)"
        )

        plt.title(
            f"Embedding Space — "
            f"Client {client} — "
            f"Round {round_number}"
        )

        plt.grid(
            alpha=0.2
        )

        plt.legend()

        plt.tight_layout()

        plt.show()